In [1]:
import os
import json
import pandas as pd
from tqdm import tqdm


from utils.prompts import generate_question_evaluation_prompt_v2
from utils.bedrock_functions import invoke_bedrock_endpoint, build_anthropic_request_body
from utils.bedrock_functions import build_mistral_request_body, build_nova_request_body, build_llama_request_body
from utils.subgraph_functions import plot_graph_with_simplified_labels, prune_triples, restore_full_triples
from utils.helper_functions import read_jsonl_file, save_as_jsonl

In [2]:
directory = "/home/ec2-user/preetam_experiments"
split = 'test'
data = json.load(open(f"{directory}/batch_processing_files/{split}/intermediate_res/qa_pairs_{split}.json"))

In [3]:
idx = 1
print(data[idx]['question'])
prompt = generate_question_evaluation_prompt_v2(data[idx]['question'])


Which broader taxonomic group includes both the guanaco and llama through their shared ancestry?


In [4]:
print(prompt)

As an expert evaluator, your role is to assess the quality and validity of trivia or natural questions. These questions aim to test the responder's knowledge, which may require implicit or external information. Your goal is to analyze the question based on the following criteria:

- **Logical Structure**: Verify if the grammar and syntax are correct. (True if grammatically and syntactically correct; False if there are issues with grammar or syntax.)
- **Redundancy**: Confirm that the question does not contain its own answer explicitly or through overly obvious phrasing. (True if it contains its answer; False if it does not.)
- **Multiple Answers**: Determine if the question allows for multiple valid answers. This is acceptable in some cases, but flag it if it reduces the question's effectiveness or specificity. (True if multiple answers are plausible; False if only one valid answer is expected.)
    
#### Output JSON Keys:
- `question`: The input question.
- `logical_structure_flag`: (

In [5]:
# data[0]

In [6]:
annotator_max_tokens = 2048
annotator_temp = 0

annotator_requests_llama = []
annotator_requests_mistral = []
annotator_requests_nova = []

for idx in tqdm(range(107)): #len(data) FOR THE LIMIT OF BATCH INFERECING
    recordId = str(data[idx]['id']).zfill(9) + str(data[idx]['question_num']).zfill(2)
    prompt = generate_question_evaluation_prompt_v2(data[idx]['question'])
    

    request_llama = build_llama_request_body(prompt=prompt, max_tokens=annotator_max_tokens, temperature=annotator_temp)
    request_mistral = build_mistral_request_body(prompt=prompt, max_tokens=annotator_max_tokens, temperature=annotator_temp)
    request_nova = build_nova_request_body(prompt=prompt, max_tokens=annotator_max_tokens, temperature=annotator_temp)
    
    annotator_requests_llama.append({'recordId': recordId, 'modelInput': request_llama})
    annotator_requests_mistral.append({'recordId': recordId, 'modelInput': request_mistral})
    annotator_requests_nova.append({'recordId': recordId, 'modelInput': request_nova})

100%|██████████| 107/107 [00:00<00:00, 25786.63it/s]


In [7]:
save_as_jsonl(annotator_requests_llama, f"{directory}/batch_processing_files/{split}/annotators/annotator_requests_qcheck_llama_{split}.jsonl") 
save_as_jsonl(annotator_requests_mistral, f"{directory}/batch_processing_files/{split}/annotators/annotator_requests_qcheck_mistral_{split}.jsonl")
save_as_jsonl(annotator_requests_nova, f"{directory}/batch_processing_files/{split}/annotators/annotator_requests_qcheck_nova_{split}.jsonl")

In [10]:
annotator_requests_mistral[7]

{'recordId': '00011888001',
 'modelInput': {'modelId': 'mistral.mistral-small-2402-v1:0',
  'contentType': 'application/json',
  'accept': 'application/json',
  'body': '{"prompt": "<s>[INST] As an expert evaluator, your role is to assess the quality and validity of trivia or natural questions. These questions aim to test the responder\'s knowledge, which may require implicit or external information. Your goal is to analyze the question based on the following criteria:\\n\\n- **Logical Structure**: Verify if the grammar and syntax are correct. (True if grammatically and syntactically correct; False if there are issues with grammar or syntax.)\\n- **Redundancy**: Confirm that the question does not contain its own answer explicitly or through overly obvious phrasing. (True if it contains its answer; False if it does not.)\\n- **Multiple Answers**: Determine if the question allows for multiple valid answers. This is acceptable in some cases, but flag it if it reduces the question\'s effec

In [7]:
annotator_max_tokens = 2048
annotator_temp = 0


request_llama = build_llama_request_body(prompt=prompt, max_tokens=annotator_max_tokens, temperature=annotator_temp)
request_mistral = build_mistral_request_body(prompt=prompt, max_tokens=annotator_max_tokens, temperature=annotator_temp)
request_nova = build_nova_request_body(prompt=prompt, max_tokens=annotator_max_tokens, temperature=annotator_temp)


In [8]:
request_llama

{'prompt': 'As an expert evaluator, your role is to assess the quality and validity of trivia or natural questions. These questions aim to test the responder\'s knowledge, which may require implicit or external information. Your goal is to analyze the question based on the following criteria:\n\n- **Logical Structure**: Verify if the grammar and syntax are correct. (True if grammatically and syntactically correct; False if there are issues with grammar or syntax.)\n- **Redundancy**: Confirm that the question does not contain its own answer explicitly or through overly obvious phrasing. (True if it contains its answer; False if it does not.)\n- **Multiple Answers**: Determine if the question allows for multiple valid answers. This is acceptable in some cases, but flag it if it reduces the question\'s effectiveness or specificity. (True if multiple answers are plausible; False if only one valid answer is expected.)\n    \n#### Output JSON Keys:\n- `question`: The input question.\n- `logi

In [9]:
json.dumps(request_llama['body'])

'"{\\"prompt\\": \\"As an expert evaluator, your role is to assess the quality and validity of trivia or natural questions. These questions aim to test the responder\'s knowledge, which may require implicit or external information. Your goal is to analyze the question based on the following criteria:\\\\n\\\\n- **Logical Structure**: Verify if the grammar and syntax are correct. (True if grammatically and syntactically correct; False if there are issues with grammar or syntax.)\\\\n- **Redundancy**: Confirm that the question does not contain its own answer explicitly or through overly obvious phrasing. (True if it contains its answer; False if it does not.)\\\\n- **Multiple Answers**: Determine if the question allows for multiple valid answers. This is acceptable in some cases, but flag it if it reduces the question\'s effectiveness or specificity. (True if multiple answers are plausible; False if only one valid answer is expected.)\\\\n    \\\\n#### Output JSON Keys:\\\\n- `question`:

In [10]:
a={"recordId": "00009785500", "modelInput": {"modelId": "us.meta.llama3-3-70b-instruct-v1:0", "contentType": "application/json", "accept": "application/json", "body": "{\"prompt\": \"As an expert evaluator, your role is to assess the quality and validity of trivia or natural questions. These questions aim to test the responder's knowledge, which may require implicit or external information. Your goal is to analyze the question based on the following criteria:\\n\\n- **Logical Structure**: Verify if the grammar and syntax are correct. (True if grammatically and syntactically correct; False if there are issues with grammar or syntax.)\\n- **Redundancy**: Confirm that the question does not contain its own answer explicitly or through overly obvious phrasing. (True if it contains its answer; False if it does not.)\\n- **Multiple Answers**: Determine if the question allows for multiple valid answers. This is acceptable in some cases, but flag it if it reduces the question's effectiveness or specificity. (True if multiple answers are plausible; False if only one valid answer is expected.)\\n    \\n#### Output JSON Keys:\\n- `question`: The input question.\\n- `logical_structure_flag`: (True/False)\\n- `logical_structure_reasoning`: Reason for the logical structure flag.\\n- `redundancy_flag`: (True/False)\\n- `redundancy_reasoning`: Reason for the redundancy flag.\\n- `multiple_answers_flag`: (True/False)\\n- `multiple_answers_reasoning`: Reason for the multiple answers flag.\\n\\n#### Task:\\nAnalyze the following question and provide a JSON object containing flags and reasons for potential issues:\\n\\n**Question**: \\\"What family does the alpaca belong to, considering its evolutionary lineage through its parent taxa?\\\"\\n\\n#### Output:\\nReturn a JSON object that evaluates the question based on the criteria above.\\n\", \"max_gen_len\": 2048, \"temperature\": 0}"}}

In [11]:
b = {
 "modelId": "meta.llama3-3-70b-instruct-v1:0",
 "contentType": "application/json",
 "accept": "application/json",
 "body": "{\"prompt\":\"this is where you place your input text\",\"max_gen_len\":512,\"temperature\":0.5,\"top_p\":0.9}"
}

In [22]:
json.loads(a['modelInput']['body'])

{'prompt': 'As an expert evaluator, your role is to assess the quality and validity of trivia or natural questions. These questions aim to test the responder\'s knowledge, which may require implicit or external information. Your goal is to analyze the question based on the following criteria:\n\n- **Logical Structure**: Verify if the grammar and syntax are correct. (True if grammatically and syntactically correct; False if there are issues with grammar or syntax.)\n- **Redundancy**: Confirm that the question does not contain its own answer explicitly or through overly obvious phrasing. (True if it contains its answer; False if it does not.)\n- **Multiple Answers**: Determine if the question allows for multiple valid answers. This is acceptable in some cases, but flag it if it reduces the question\'s effectiveness or specificity. (True if multiple answers are plausible; False if only one valid answer is expected.)\n    \n#### Output JSON Keys:\n- `question`: The input question.\n- `logi

In [23]:
json.loads(b['body'])

{'prompt': 'this is where you place your input text',
 'max_gen_len': 512,
 'temperature': 0.5,
 'top_p': 0.9}

In [17]:
list(a['modelInput'].keys()) == list(b.keys())

True

In [8]:
res = read_jsonl_file('/home/ec2-user/preetam_experiments/batch_processing_files/test/annotators/results/annotator_requests_qcheck_llama_testOut.jsonl')

In [10]:
res[0]#['modelInput']

{'modelInput': '{"prompt": "As an expert evaluator, your role is to assess the quality and validity of trivia or natural questions. These questions aim to test the responder\'s knowledge, which may require implicit or external information. Your goal is to analyze the question based on the following criteria:\\n\\n- **Logical Structure**: Verify if the grammar and syntax are correct. (True if grammatically and syntactically correct; False if there are issues with grammar or syntax.)\\n- **Redundancy**: Confirm that the question does not contain its own answer explicitly or through overly obvious phrasing. (True if it contains its answer; False if it does not.)\\n- **Multiple Answers**: Determine if the question allows for multiple valid answers. This is acceptable in some cases, but flag it if it reduces the question\'s effectiveness or specificity. (True if multiple answers are plausible; False if only one valid answer is expected.)\\n    \\n#### Output JSON Keys:\\n- `question`: The i

In [9]:
# save_as_jsonl(llm_requests_list, f"/home/ec2-user/preetam_experiments/batch_processing_files/{split}/llm_requests_{split}.jsonl") 
# #f"{directory}/llm_requests_{split}.jsonl"


In [9]:
res_file = read_jsonl_file(f"llm_requests_{split}.jsonl.out")

In [12]:
res_file[0]['modelOutput']

{'id': 'msg_bdrk_01V58UM7i7VXVVnFiiUH4fNS',
 'type': 'message',
 'role': 'assistant',
 'model': 'claude-3-5-sonnet-20241022',
 'content': [{'type': 'text',
   'text': '{\n    "valid_qa_pairs": true,\n    "number_of_qa_pairs": 3,\n    "qa_pairs": [\n        {\n            "question": "What family of animals includes both the guanaco and llama?",\n            "answer": "Camelidae",\n            "supporting_path": [\n                {\n                    "subject": "Guanaco",\n                    "predicate": "parentTaxon",\n                    "object": "Lama__u0028_genus_u0029_"\n                },\n                {\n                    "subject": "Lama__u0028_genus_u0029_",\n                    "predicate": "parentTaxon",\n                    "object": "Camelidae"\n                },\n                {\n                    "subject": "Llama",\n                    "predicate": "parentTaxon",\n                    "object": "Lama__u0028_genus_u0029_"\n                }\n            ]\n 

In [50]:
df.iloc[99]['subgraph_Steiner_largest_connected']

[['http://yago-knowledge.org/resource/Mazda_B_series',
  'http://schema.org/manufacturer',
  'http://yago-knowledge.org/resource/Mazda'],
 ['http://yago-knowledge.org/resource/Troller_T4',
  'http://schema.org/manufacturer',
  'http://yago-knowledge.org/resource/Troller_Veículos_Especiais'],
 ['http://yago-knowledge.org/resource/Ford_Ranger',
  'http://schema.org/manufacturer',
  'http://yago-knowledge.org/resource/Ford_Motor_Company'],
 ['http://yago-knowledge.org/resource/Ford_Transit',
  'http://schema.org/manufacturer',
  'http://yago-knowledge.org/resource/Ford_Motor_Company'],
 ['http://yago-knowledge.org/resource/Maurice_Jordan_Q3300959',
  'http://schema.org/worksFor',
  'http://yago-knowledge.org/resource/PSA_Group'],
 ['http://yago-knowledge.org/resource/Maurice_Jordan_Q3300959',
  'http://schema.org/worksFor',
  'http://yago-knowledge.org/resource/Peugeot'],
 ['http://yago-knowledge.org/resource/1979_World_Rally_Championship_For_Manufacturers_Q20202921',
  'http://yago-knowl

In [47]:

resp = json.loads(res_file[99]['modelOutput']['content'][0]['text'])
resp

{'valid_qa_pairs': True,
 'number_of_qa_pairs': 3,
 'qa_pairs': [{'question': 'Which car manufacturer, who owns a company that makes the T4, participated in the 1979 World Rally Championship?',
   'answer': 'Ford_Motor_Company',
   'supporting_path': [{'subject': 'Troller_T4',
     'predicate': 'manufacturer',
     'object': 'Troller_Veículos_Especiais'},
    {'subject': 'Troller_Veículos_Especiais',
     'predicate': 'ownedBy',
     'object': 'Ford_Motor_Company'},
    {'subject': '1979_World_Rally_Championship_For_Manufacturers_Q20202921',
     'predicate': 'participant',
     'object': 'Ford_Motor_Company'}]},
  {'question': 'Which company that Maurice Jordan worked for was also a participant in the 1979 World Rally Championship?',
   'answer': 'Peugeot',
   'supporting_path': [{'subject': 'Maurice_Jordan_Q3300959',
     'predicate': 'worksFor',
     'object': 'Peugeot'},
    {'subject': '1979_World_Rally_Championship_For_Manufacturers_Q20202921',
     'predicate': 'participant',
  

In [55]:
print(resp['qa_pairs'][0]['question'])
print(resp['qa_pairs'][0]['answer'])
resp['qa_pairs'][0]['supporting_path']
restore_full_triples_universal(resp['qa_pairs'][0]['supporting_path'], df.iloc[99]['subgraph_Steiner_largest_connected'])
graph = crea

Which car manufacturer, who owns a company that makes the T4, participated in the 1979 World Rally Championship?
Ford_Motor_Company


([['http://yago-knowledge.org/resource/Troller_T4',
   'http://schema.org/manufacturer',
   'http://yago-knowledge.org/resource/Troller_Veículos_Especiais'],
  ['http://yago-knowledge.org/resource/Troller_Veículos_Especiais',
   'http://yago-knowledge.org/resource/ownedBy',
   'http://yago-knowledge.org/resource/Ford_Motor_Company'],
  ['http://yago-knowledge.org/resource/1979_World_Rally_Championship_For_Manufacturers_Q20202921',
   'http://yago-knowledge.org/resource/participant',
   'http://yago-knowledge.org/resource/Ford_Motor_Company']],
 True)

In [18]:
llm_requests_list[0]['modelInput']

{'anthropic_version': 'bedrock-2023-05-31',
 'system': 'You are helpful assistant.',
 'messages': [{'role': 'user',
   'content': "You are an AI assistant tasked with generating question-answer pairs from knowledge graph triples. Your goal is to create natural, human-like questions and their corresponding answers based on the provided graph data.\n\nTask Overview:\nGenerate **multi-hop, complex Q&A pairs** where the questions appear simple and natural but require reasoning across multiple connected relationships within the graph to infer the answer.\n\nGuidelines for Generating Q&A Pairs:\n1. **Question Design**:\n- Questions should utilize multiple connected relationships in the graph, requiring multi-hop reasoning.\n- Avoid single-hop or trivial questions directly derived from a single triple.\n- The answer should be an entity or node in the graph.\n\n2. **Multi-Hop Reasoning**:\n- Use paths connecting entities indirectly through multiple relationships to infer answers.\n- Questions 

In [19]:

model_id = "us.anthropic.claude-3-5-sonnet-20241022-v2:0"
response_data = invoke_bedrock_endpoint(llm_requests_list[0]['modelInput'], model_id)
print("Response from Claude:", json.dumps(response_data, indent=2))

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


Response from Claude: {
  "id": "msg_bdrk_0135t5PoReNfhHVQYXTz6SDQ",
  "type": "message",
  "role": "assistant",
  "model": "claude-3-5-sonnet-20241022",
  "content": [
    {
      "type": "text",
      "text": "{\n    \"valid_qa_pairs\": true,\n    \"number_of_qa_pairs\": 3,\n    \"qa_pairs\": [\n        {\n            \"question\": \"What family of animals includes both the alpaca and guanaco, despite them belonging to different genera?\",\n            \"answer\": \"Camelidae\",\n            \"supporting_path\": [\n                {\n                    \"subject\": \"Alpaca\",\n                    \"predicate\": \"parentTaxon\",\n                    \"object\": \"Vicugna\"\n                },\n                {\n                    \"subject\": \"Vicugna\",\n                    \"predicate\": \"parentTaxon\",\n                    \"object\": \"Camelidae\"\n                },\n                {\n                    \"subject\": \"Guanaco\",\n                    \"predicate\": \"paren

In [28]:
resp = json.loads(response_data['content'][0]['text'])
restore_full_triples_universal(resp['qa_pairs'][0]['supporting_path'], df.iloc[0]['subgraph_Steiner_largest_connected'])

([['http://yago-knowledge.org/resource/Alpaca',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Vicugna'],
  ['http://yago-knowledge.org/resource/Vicugna',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Camelidae'],
  ['http://yago-knowledge.org/resource/Guanaco',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Lama__u0028_genus_u0029_'],
  ['http://yago-knowledge.org/resource/Lama__u0028_genus_u0029_',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Camelidae']],
 True)

In [27]:
json.loads(response_data['content'][0]['text'])

{'valid_qa_pairs': True,
 'number_of_qa_pairs': 3,
 'qa_pairs': [{'question': 'What family of animals includes both the alpaca and guanaco, despite them belonging to different genera?',
   'answer': 'Camelidae',
   'supporting_path': [{'subject': 'Alpaca',
     'predicate': 'parentTaxon',
     'object': 'Vicugna'},
    {'subject': 'Vicugna', 'predicate': 'parentTaxon', 'object': 'Camelidae'},
    {'subject': 'Guanaco',
     'predicate': 'parentTaxon',
     'object': 'Lama__u0028_genus_u0029_'},
    {'subject': 'Lama__u0028_genus_u0029_',
     'predicate': 'parentTaxon',
     'object': 'Camelidae'}]},
  {'question': 'Which higher-level taxonomic group connects both the Chevrotain and the Pecora through their evolutionary relationships?',
   'answer': 'Ruminant',
   'supporting_path': [{'subject': 'Chevrotain',
     'predicate': 'parentTaxon',
     'object': 'Tragulina'},
    {'subject': 'Tragulina', 'predicate': 'parentTaxon', 'object': 'Ruminant'},
    {'subject': 'Pecora', 'predicate'

In [6]:
request_json = build_anthropic_request_body(
            system_prompt="You are helpful assistant.",
            user_prompt=prompt,
            max_tokens=1024,
            temperature=0
        )


model_id = "us.anthropic.claude-3-5-sonnet-20241022-v2:0"

        # Invoke the endpoint.
response_data = invoke_bedrock_endpoint(request_json, model_id)
print("Response from Claude:", json.dumps(response_data, indent=2))

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


Response from Claude: {
  "id": "msg_bdrk_01LeoVUs4q5o9TxWbnHnRwbP",
  "type": "message",
  "role": "assistant",
  "model": "claude-3-5-sonnet-20241022",
  "content": [
    {
      "type": "text",
      "text": "{\n    \"valid_qa_pairs\": true,\n    \"number_of_qa_pairs\": 3,\n    \"qa_pairs\": [\n        {\n            \"question\": \"What family of animals includes both the alpaca and the guanaco, despite them belonging to different genera?\",\n            \"answer\": \"Camelidae\",\n            \"supporting_path\": [\n                {\n                    \"subject\": \"Alpaca\",\n                    \"predicate\": \"parentTaxon\",\n                    \"object\": \"Vicugna\"\n                },\n                {\n                    \"subject\": \"Vicugna\",\n                    \"predicate\": \"parentTaxon\",\n                    \"object\": \"Camelidae\"\n                },\n                {\n                    \"subject\": \"Guanaco\",\n                    \"predicate\": \"p

In [8]:
resp = json.loads(response_data['content'][0]['text'])

In [1]:
resp

NameError: name 'resp' is not defined

In [13]:
lenjson.dumps(request_json)

'{"anthropic_version": "bedrock-2023-05-31", "system": "You are helpful assistant.", "messages": [{"role": "user", "content": "\\nYou are an AI assistant tasked with generating question-answer pairs from knowledge graph triples. Your goal is to create natural, human-like questions and their corresponding answers based on the provided graph data.\\n\\nTask Overview:\\nGenerate **multi-hop, complex Q&A pairs** where the questions appear simple and natural but require reasoning across multiple connected relationships within the graph to infer the answer.\\n\\nGuidelines for Generating Q&A Pairs:\\n1. **Question Design**:\\n- Questions should utilize multiple connected relationships in the graph, requiring multi-hop reasoning.\\n- Avoid single-hop or trivial questions directly derived from a single triple.\\n- The answer should be an entity or node in the graph.\\n\\n2. **Multi-Hop Reasoning**:\\n- Use paths connecting entities indirectly through multiple relationships to infer answers.\\n

In [9]:
restore_full_triples_universal(resp['qa_pairs'][0]['supporting_path'], df.iloc[idx]['subgraph_Steiner_largest_connected'])

([['http://yago-knowledge.org/resource/Alpaca',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Vicugna'],
  ['http://yago-knowledge.org/resource/Vicugna',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Camelidae'],
  ['http://yago-knowledge.org/resource/Guanaco',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Lama__u0028_genus_u0029_'],
  ['http://yago-knowledge.org/resource/Lama__u0028_genus_u0029_',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Camelidae']],
 True)

In [25]:
df.iloc[idx]['subgraph_Steiner_largest_connected']

[['http://yago-knowledge.org/resource/Camelidae',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Even-toed_ungulate'],
 ['http://yago-knowledge.org/resource/Even-toed_ungulate',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Ungulate'],
 ['http://yago-knowledge.org/resource/Tragulina',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Ruminant'],
 ['http://yago-knowledge.org/resource/Chevrotain',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Even-toed_ungulate'],
 ['http://yago-knowledge.org/resource/Chevrotain',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Tragulina'],
 ['http://yago-knowledge.org/resource/Camel',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Camelidae'],
 ['http://yago-knowledge.org/resource/Guanaco',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Lama__u0028_genus_u0029_'],
 ['http://ya

In [26]:
resp['qa_pairs'][0]['supporting_path']

[{'subject': 'Vicuña', 'predicate': 'parentTaxon', 'object': 'Vicugna'},
 {'subject': 'Vicugna', 'predicate': 'parentTaxon', 'object': 'Camelidae'},
 {'subject': 'Guanaco',
  'predicate': 'parentTaxon',
  'object': 'Lama__u0028_genus_u0029_'},
 {'subject': 'Lama__u0028_genus_u0029_',
  'predicate': 'parentTaxon',
  'object': 'Camelidae'}]